In [25]:
import pandas as pd
import numpy as np

# Ruta del archivo
file_path = "OrderAllocation_v2.csv"

# Cargar el CSV en un DataFrame
df = pd.read_csv(file_path)

# Mostrar las primeras filas del DataFrame
df.head()


,orderLine.order.orderIdentifier,orderLine.orderLineNumber,productItem.partNumber,quantityRequired,quantityRequiredUnits,quantityAllocated,quantityAllocatedUnits,allocationComment,alternateItems,sourceLink
0,100044332,100,45000223,56,ea,54,ea,"2020-08-22, 0",NaN,http://lighttree.com/100044332
1,100044332,100,2530020,56,ea,53,ea,"2020-08-20, 25","2530003, 8220072",http://lighttree.com/100044332
2,100044332,100,39000221,56,ea,56,ea,NaN,39000224,http://lighttree.com/100044332
3,100044332,100,29000462,56,ea,56,ea,NaN,29000469,http://lighttree.com/100044332
4,100044332,100,83600300,56,ea,56,ea,NaN,83600200,http://lighttree.com/100044332


In [26]:
# Separar la columna allocationComment en dos nuevas columnas
df[['allocationDate', 'allocationValue']] = df['allocationComment'].str.split(',', expand=True)

# Convertir allocationDate a formato de fecha y allocationValue a número
df['allocationDate'] = pd.to_datetime(df['allocationDate'], errors='coerce')
df['allocationValue'] = pd.to_numeric(df['allocationValue'], errors='coerce')

# Convertir la columna allocationValue a un array de tipo float
allocation_values = df['allocationValue'].to_numpy(dtype=float)

# 1. np.isnan(arr) - Detectar valores NaN
nan_mask = np.isnan(allocation_values)  # Devuelve un array booleano donde hay NaN
#print("¿Dónde hay valores NaN?:", nan_mask)

# Contar cuántos valores nulos hay
num_nulls = np.sum(nan_mask)

#print("\nCantidad de valores nulos en la columna Allocation_Values:", num_nulls)

allocation_stats = {
    "mean": df["allocationValue"].mean(),
    "median": df["allocationValue"].median(),
    "min": df["allocationValue"].min(),
    "max": df["allocationValue"].max(),
    "std": df["allocationValue"].std()
}

print(allocation_stats)

{'mean': np.float64(569.0555555555555), 'median': np.float64(26.0), 'min': np.float64(0.0), 'max': np.float64(10603.0), 'std': np.float64(2014.8102774128026)}


In [30]:
# Calcular Q1 y Q3
Q1 = df["allocationValue"].quantile(0.25)
Q3 = df["allocationValue"].quantile(0.75)

# Calcular IQR
IQR = Q3 - Q1

# Definir límites para outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filtrar valores dentro de los límites
df_filtered = df[(df["allocationValue"] >= lower_bound) & (df["allocationValue"] <= upper_bound)]

# Recalcular estadísticas después de eliminar outliers
allocation_stats_filtered = {
    "mean": df_filtered["allocationValue"].mean(),
    "median": df_filtered["allocationValue"].median(),
    "min": df_filtered["allocationValue"].min(),
    "max": df_filtered["allocationValue"].max(),
    "std": df_filtered["allocationValue"].std()
}

print(allocation_stats_filtered)

# Rellenar valores nulos en allocationValue con la nueva media calculada
df_filtered.loc[:, "allocationValue"].fillna(allocation_stats_filtered["mean"], inplace=True)

# Verificar que ya no haya valores nulos en la columna
null_count_after_fill = df_filtered["allocationValue"].isnull().sum()

print(df_filtered["allocationValue"], null_count_after_fill)

{'mean': np.float64(59.58064516129032), 'median': np.float64(14.0), 'min': np.float64(0.0), 'max': np.float64(309.0), 'std': np.float64(88.60315050589281)}
0        0.0
1       25.0
15     301.0
16      14.0
30       8.0
31       0.0
45       4.0
46     309.0
60      10.0
61      14.0
75      73.0
76     142.0
78     104.0
79     175.0
93      27.0
94       2.0
108      2.0
109      5.0
123      8.0
124      6.0
138     12.0
139     60.0
156     28.0
157    102.0
171    252.0
172     42.0
186      0.0
187      1.0
201      0.0
202     22.0
216     99.0
Name: allocationValue, dtype: float64 0


/tmp/ipykernel_2100/2294951656.py:27: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_filtered.loc[:, "allocationValue"].fillna(allocation_stats_filtered["mean"], inplace=True)
/tmp/ipykernel_2100/2294951656.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered.loc[:, "allocationValue"].fillna(allocation_stats_filtered["mean"], inp

In [31]:
# Crear la columna allocationDifference
df_filtered.loc[:, "allocationDifference"] = df_filtered["quantityRequired"] - df_filtered["quantityAllocated"]

# Extraer información de allocationDate
df_filtered.loc[:, "allocationYear"] = df_filtered["allocationDate"].dt.year
df_filtered.loc[:, "allocationMonth"] = df_filtered["allocationDate"].dt.month
df_filtered.loc[:, "allocationWeekday"] = df_filtered["allocationDate"].dt.day_name()

# Mostrar algunas filas con las nuevas columnas
df_filtered[["quantityRequired", "quantityAllocated", "allocationDifference", 
             "allocationDate", "allocationYear", "allocationMonth", "allocationWeekday"]].head()

/tmp/ipykernel_2100/1651144483.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered.loc[:, "allocationDifference"] = df_filtered["quantityRequired"] - df_filtered["quantityAllocated"]
/tmp/ipykernel_2100/1651144483.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered.loc[:, "allocationYear"] = df_filtered["allocationDate"].dt.year
/tmp/ipykernel_2100/1651144483.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,c

,quantityRequired,quantityAllocated,allocationDifference,allocationDate,allocationYear,allocationMonth,allocationWeekday
0,56,54,2,2020-08-22,2020,8,Saturday
1,56,53,3,2020-08-20,2020,8,Thursday
15,512,470,42,2020-08-20,2020,8,Thursday
16,512,509,3,2020-08-22,2020,8,Saturday
30,212,210,2,2020-08-29,2020,8,Saturday
